# Defensive Cybersecurity 50M: first-run pilot

This starts a small from-scratch decoder-only Transformer on one Kaggle GPU. The corpus is intentionally defensive and small; this 100-step run is a pipeline pilot and will not yield broad cybersecurity competence. The private input contains redacted Human-only tone samples.

In [ ]:
%pip -q install pypdf tokenizers
import os, shutil, subprocess, sys
repo = '/kaggle/working/defensive-cyber-50m'
if os.path.exists(repo): shutil.rmtree(repo)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Jether34/G3-4B-Q_k4-finetune-project-.git', repo], check=True)
os.chdir(repo)
print('GPU:', subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))

In [ ]:
# Fetch the small official defensive references and build a local JSONL corpus.
subprocess.run([sys.executable, 'cyber_50m/fetch_reference_data.py'], check=True)
subprocess.run([sys.executable, 'cyber_50m/prepare_official_text.py'], check=True)
os.makedirs('data/derived', exist_ok=True)
style_files = [p for p in __import__('glob').glob('/kaggle/input/*/copilot_style_private.jsonl')]
if style_files: shutil.copy2(style_files[0], 'data/derived/copilot_style_private.jsonl')
else: print('No private tone dataset attached; running defensive-text-only pilot.')
print('Prepared documents:', subprocess.check_output([sys.executable, '-c', "import json; print(sum(1 for _ in open('data/derived/official_defensive_text.jsonl',encoding='utf-8')))"], text=True).strip())

In [ ]:
# Start a bounded first run; checkpoints and metrics are written to Kaggle output storage.
subprocess.run([sys.executable, 'cyber_50m/train_gpt.py', '--steps', '100', '--batch-size', '2', '--grad-accum', '4', '--out-dir', '/kaggle/working/cyber_50m_pilot'], check=True)

In [ ]:
import json, os
summary_path = '/kaggle/working/cyber_50m_pilot/run_summary.json'
print(open(summary_path, encoding='utf-8').read())
print('Artifacts:', os.listdir('/kaggle/working/cyber_50m_pilot'))